# L5: Memory Aware Agent

<div style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> <p>⏳ <b>Note <code>(Database Starting)</code>:</b> This notebook takes about 30-60 seconds to be ready to use. You may start and watch the video while you wait.</p>
<p>If you see <tt>Admin connection failed</tt> after running the first cell, simply wait and re-run — it is not a credentials issue.</p>
</div>

This lesson brings together everything from the previous labs to build a complete **Memory Aware Agent**—an AI system that can remember past conversations, learn from interactions, and manage its context window intelligently.

**Lesson Objectives**

By the end of this lesson, you will understand how to:
- Integrate all memory types (conversational, semantic, workflow, entity, summary, tool logs) into a unified agent
- Implement context window management with automatic summarization
- Build an agent loop that retrieves relevant context before each response
- Use Just-In-Time (JIT) retrieval to expand summaries on demand

**Key Concepts**

| Concept | Description |
|---------|-------------|
| **Memory Aware Agent** | An agent that reads from and writes to persistent memory stores during execution |
| **Context Engineering** | Dynamically building the optimal context window for each query |
| **Just-In-Time Retrieval** | Fetching detailed information only when the agent needs it |
| **Automatic Summarization** | Compressing context when usage exceeds thresholds |

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> and <code>helper.py</code> files:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>.

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download as"</em> and select <em>"Notebook (.ipynb)"</em>.</p>

</div>

## Part 1: Setup and Infrastructure

This section initializes the complete memory infrastructure needed for our agent. We reuse the components built in previous labs:
- **Database Connection** — Oracle Database for persistent storage
- **Memory Stores** — Vector stores for each memory type
- **MemoryManager** — Unified interface for memory operations
- **Toolbox** — Registry for agent-callable tools

### Database and Embedding Setup

In [ ]:
from helper import suppress_warnings
suppress_warnings()

from helper import load_env, setup_oracle_database, connect_to_oracle
import oracledb
import os

# Load env variables
load_env()

# Read from env
SYS_USER = os.getenv("ORACLE_SYS_USER")
SYS_PASSWORD = os.getenv("ORACLE_SYS_PASSWORD")
VECTOR_USER = os.getenv("ORACLE_VECTOR_USER")
VECTOR_PASSWORD = os.getenv("ORACLE_VECTOR_PASSWORD")
DSN = os.getenv("ORACLE_DSN")

# Step 1: Connect as SYS
admin_conn = oracledb.connect(
    user=SYS_USER,
    password=SYS_PASSWORD,
    dsn=DSN,
    mode=oracledb.SYSDBA
)

print("✅ Connected as SYS")

# Step 2: Setup
setup_oracle_database(
    admin_user=SYS_USER,
    admin_password=SYS_PASSWORD,
    dsn=DSN,
    vector_password=VECTOR_PASSWORD
)

# Step 3: Connect as VECTOR
database_connection = connect_to_oracle(
    user=VECTOR_USER,
    password=VECTOR_PASSWORD,
    dsn=DSN
)

print("✅ Connected as:", database_connection.username)

In [2]:
import os
from dotenv import load_dotenv
import google.generativeai as genai
from langchain_community.embeddings import HuggingFaceEmbeddings

# Load API key
load_dotenv()
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

# Gemini model (LLM replacement)
model = genai.GenerativeModel("gemini-2.5-flash-lite")

# Embeddings (unchanged)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-mpnet-base-v2"
)

In [3]:
import os
from dotenv import load_dotenv
from google import genai
from helper import (
    GeminiLLM,
    Toolbox,
    load_env,
    connect_to_oracle,
    setup_oracle_database,
    # MemoryManager   # 👈 make sure this exists
)

load_dotenv()

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
llm = GeminiLLM(client, model="gemini-2.5-flash-lite")
# toolbox = Toolbox(memory_manager, llm, embedding_model)

In [4]:
# changed, dont run


# from openai import OpenAI
# from langchain_community.embeddings import HuggingFaceEmbeddings

# client = OpenAI()

# # Initialize the embedding model
# embedding_model = HuggingFaceEmbeddings(
#     model_name="sentence-transformers/paraphrase-mpnet-base-v2"
# )

In [5]:
def create_banking_tables(db_connection):
    """Create banking-specific tables in Oracle"""
    try:
        with db_connection.cursor() as cur:
            
            # Client Masters Table
            cur.execute("""
                CREATE TABLE IF NOT EXISTS client_masters (
                    client_id VARCHAR2(50) PRIMARY KEY,
                    name VARCHAR2(200),
                    email VARCHAR2(100),
                    phone VARCHAR2(20),
                    date_of_birth DATE,
                    client_since DATE,
                    net_worth NUMBER(15,2),
                    risk_profile VARCHAR2(50),
                    status VARCHAR2(20) DEFAULT 'ACTIVE',
                    created_at TIMESTAMP DEFAULT SYSTIMESTAMP
                )
            """)
            
            # Client Family Information
            cur.execute("""
                CREATE TABLE IF NOT EXISTS client_family (
                    family_id VARCHAR2(50) PRIMARY KEY,
                    client_id VARCHAR2(50),
                    name VARCHAR2(200),
                    relationship VARCHAR2(50),
                    age NUMBER(3),
                    is_dependent CHAR(1),
                    created_at TIMESTAMP DEFAULT SYSTIMESTAMP,
                    FOREIGN KEY (client_id) REFERENCES client_masters(client_id)
                )
            """)
            
            # Client Assets
            cur.execute("""
                CREATE TABLE IF NOT EXISTS client_assets (
                    asset_id VARCHAR2(50) PRIMARY KEY,
                    client_id VARCHAR2(50),
                    asset_type VARCHAR2(100),
                    value NUMBER(15,2),
                    description VARCHAR2(500),
                    status VARCHAR2(20) DEFAULT 'ACTIVE',
                    last_updated TIMESTAMP DEFAULT SYSTIMESTAMP,
                    FOREIGN KEY (client_id) REFERENCES client_masters(client_id)
                )
            """)
            
            # Client Financial Goals
            cur.execute("""
                CREATE TABLE IF NOT EXISTS client_financial_goals (
                    goal_id VARCHAR2(50) PRIMARY KEY,
                    client_id VARCHAR2(50),
                    goal_name VARCHAR2(200),
                    target_amount NUMBER(15,2),
                    timeline VARCHAR2(100),
                    priority VARCHAR2(50),
                    status VARCHAR2(20) DEFAULT 'ACTIVE',
                    created_at TIMESTAMP DEFAULT SYSTIMESTAMP,
                    FOREIGN KEY (client_id) REFERENCES client_masters(client_id)
                )
            """)
            
            # Client Life Events
            cur.execute("""
                CREATE TABLE IF NOT EXISTS client_life_events (
                    event_id VARCHAR2(50) PRIMARY KEY,
                    client_id VARCHAR2(50),
                    event_date DATE,
                    event_type VARCHAR2(100),
                    description VARCHAR2(500),
                    impact VARCHAR2(200),
                    created_at TIMESTAMP DEFAULT SYSTIMESTAMP,
                    FOREIGN KEY (client_id) REFERENCES client_masters(client_id)
                )
            """)
            
            # Conversation Memory (with client association)
            cur.execute("""
                CREATE TABLE IF NOT EXISTS conversation_memory (
                    id VARCHAR2(50) PRIMARY KEY,
                    client_id VARCHAR2(50),
                    thread_id VARCHAR2(50),
                    role VARCHAR2(20),
                    content CLOB,
                    timestamp TIMESTAMP DEFAULT SYSTIMESTAMP,
                    FOREIGN KEY (client_id) REFERENCES client_masters(client_id)
                )
            """)
            
            # Client Threads
            cur.execute("""
                CREATE TABLE IF NOT EXISTS client_threads (
                    thread_id VARCHAR2(50) PRIMARY KEY,
                    client_id VARCHAR2(50),
                    created_at TIMESTAMP DEFAULT SYSTIMESTAMP,
                    last_activity TIMESTAMP DEFAULT SYSTIMESTAMP,
                    status VARCHAR2(20) DEFAULT 'ACTIVE',
                    summary VARCHAR2(500),
                    FOREIGN KEY (client_id) REFERENCES client_masters(client_id)
                )
            """)
            
            # Bank Investment Schemes
            cur.execute("""
                CREATE TABLE IF NOT EXISTS investment_schemes (
                    scheme_id VARCHAR2(50) PRIMARY KEY,
                    scheme_name VARCHAR2(200),
                    description VARCHAR2(500),
                    minimum_investment NUMBER(15,2),
                    expected_return VARCHAR2(50),
                    risk_level VARCHAR2(50),
                    eligibility VARCHAR2(500),
                    status VARCHAR2(20) DEFAULT 'ACTIVE',
                    created_at TIMESTAMP DEFAULT SYSTIMESTAMP
                )
            """)
            
            # Bank Investment Products
            cur.execute("""
                CREATE TABLE IF NOT EXISTS investment_products (
                    product_id VARCHAR2(50) PRIMARY KEY,
                    product_name VARCHAR2(200),
                    product_type VARCHAR2(100),
                    description VARCHAR2(500),
                    interest_rate NUMBER(5,2),
                    term VARCHAR2(50),
                    min_amount NUMBER(15,2),
                    status VARCHAR2(20) DEFAULT 'ACTIVE',
                    created_at TIMESTAMP DEFAULT SYSTIMESTAMP
                )
            """)
            
            # Bank Credit Products
            cur.execute("""
                CREATE TABLE IF NOT EXISTS credit_products (
                    product_id VARCHAR2(50) PRIMARY KEY,
                    product_name VARCHAR2(200),
                    product_type VARCHAR2(100),
                    description VARCHAR2(500),
                    interest_rate NUMBER(5,2),
                    max_amount NUMBER(15,2),
                    eligibility VARCHAR2(500),
                    status VARCHAR2(20) DEFAULT 'ACTIVE',
                    created_at TIMESTAMP DEFAULT SYSTIMESTAMP
                )
            """)
            
            # Bank Policies
            cur.execute("""
                CREATE TABLE IF NOT EXISTS bank_policies (
                    policy_id VARCHAR2(50) PRIMARY KEY,
                    policy_name VARCHAR2(200),
                    policy_category VARCHAR2(100),
                    policy_text CLOB,
                    last_updated TIMESTAMP DEFAULT SYSTIMESTAMP,
                    status VARCHAR2(20) DEFAULT 'ACTIVE'
                )
            """)
            
            # Fee Structure
            cur.execute("""
                CREATE TABLE IF NOT EXISTS fee_structure (
                    fee_id VARCHAR2(50) PRIMARY KEY,
                    fee_type VARCHAR2(100),
                    amount NUMBER(10,2),
                    description VARCHAR2(200),
                    applicable_to VARCHAR2(200),
                    status VARCHAR2(20) DEFAULT 'ACTIVE',
                    created_at TIMESTAMP DEFAULT SYSTIMESTAMP
                )
            """)
            
            # Interest Rates
            cur.execute("""
                CREATE TABLE IF NOT EXISTS interest_rates (
                    rate_id VARCHAR2(50) PRIMARY KEY,
                    rate_type VARCHAR2(100),
                    current_rate NUMBER(5,2),
                    effective_date DATE,
                    status VARCHAR2(20) DEFAULT 'ACTIVE',
                    created_at TIMESTAMP DEFAULT SYSTIMESTAMP
                )
            """)
            
            # Service Tiers
            cur.execute("""
                CREATE TABLE IF NOT EXISTS service_tiers (
                    tier_id VARCHAR2(50) PRIMARY KEY,
                    tier_name VARCHAR2(100),
                    benefits CLOB,
                    min_portfolio_value NUMBER(15,2),
                    features CLOB,
                    status VARCHAR2(20) DEFAULT 'ACTIVE',
                    created_at TIMESTAMP DEFAULT SYSTIMESTAMP
                )
            """)
            
            db_connection.commit()
            print("✅ All banking tables created successfully")
            
    except Exception as e:
        if "already exists" in str(e).lower():
            print("✅ Tables already exist")
        else:
            logger.error(f"Error creating tables: {e}")

In [6]:
def seed_sample_data(db_connection):
    """Seed database with sample bank and client data for testing"""
    print("\n[STEP 3] Seeding Sample Data...")
    
    try:
        with db_connection.cursor() as cur:
            
            # Sample client
            client_id = "CLIENT001"
            cur.execute("""
                INSERT INTO client_masters 
                (client_id, name, email, phone, date_of_birth, client_since, net_worth, risk_profile)
                VALUES (:client_id, :name, :email, :phone, :dob, :since, :net_worth, :risk)
            """, {
                "client_id": client_id,
                "name": "Rajesh Kumar",
                "email": "rajesh.kumar@email.com",
                "phone": "9876543210",
                "dob": datetime(1970, 5, 15),
                "since": datetime(2015, 3, 20),
                "net_worth": 5000000,  # 50 lakhs
                "risk": "Moderate"
            })
            
            # Sample family members
            cur.execute("""
                INSERT INTO client_family
                (family_id, client_id, name, relationship, age, is_dependent)
                VALUES (:fid, :cid, :name, :rel, :age, :dep)
            """, {
                "fid": str(uuid.uuid4()),
                "cid": client_id,
                "name": "Priya Kumar",
                "rel": "Spouse",
                "age": 42,
                "dep": "N"
            })
            
            # Sample assets
            cur.execute("""
                INSERT INTO client_assets
                (asset_id, client_id, asset_type, value, description)
                VALUES (:aid, :cid, :type, :value, :desc)
            """, {
                "aid": str(uuid.uuid4()),
                "cid": client_id,
                "type": "Equity Portfolio",
                "value": 2000000,
                "desc": "Diversified equity holdings"
            })
            
            # Sample investment scheme
            cur.execute("""
                INSERT INTO investment_schemes
                (scheme_id, scheme_name, description, minimum_investment, 
                 expected_return, risk_level, eligibility)
                VALUES (:sid, :name, :desc, :min_inv, :ret, :risk, :elig)
            """, {
                "sid": str(uuid.uuid4()),
                "name": "WealthBank Premium Investment Plan",
                "desc": "Diversified portfolio of equities and bonds for long-term wealth creation",
                "min_inv": 100000,
                "ret": "12-15% per annum",
                "risk": "Moderate",
                "elig": "Individual investors with net worth above 10 lakhs"
            })
            
            db_connection.commit()
            print("✅ Sample data seeded successfully")
            
    except Exception as e:
        if "unique constraint" in str(e).lower():
            print("✅ Sample data already exists")
        else:
            print(f"❌ Error seeding data: {e}")

In [7]:
from datetime import datetime

In [8]:
create_banking_tables(database_connection)

✅ All banking tables created successfully


In [9]:
seed_sample_data(database_connection)


[STEP 3] Seeding Sample Data...
✅ Sample data already exists


In [10]:
class BankingWealthAgent:
    """
    Memory-aware agent for bank wealth management team interactions.
    
    Features:
    - Client-specific context building
    - Persistent conversation memory
    - Bank knowledge integration (schemes, products, policies)
    - Thread-based conversation management
    - Conversation history retrieval and summarization
    """
    
    def __init__(self, db_connection, openai_client):
        """Initialize the banking agent"""
        self.db_connection = db_connection
        self.openai_client = openai_client
        self.model = "gpt-4-turbo"  # Use latest available model
        self.current_client_id = None
        self.current_thread_id = None
        self.system_prompt = self._build_system_prompt()
        
    def _build_system_prompt(self) -> str:
        """Build the system prompt for the agent"""
        return """You are an expert wealth management advisor for a premium banking institution.
 
Your role is to:
1. Understand each client's unique financial situation, family circumstances, and life goals
2. Provide personalized wealth management advice aligned with their risk profile
3. Recommend appropriate bank schemes, investment products, and services
4. Explain bank policies, fees, and compliance requirements clearly
5. Maintain awareness of major life events and how they impact financial planning
6. Build long-term relationships through consistent, ethical advice
 
Guidelines:
- Always personalize recommendations based on client risk profile
- Consider family situation and dependents in all suggestions
- Explain pros/cons of different schemes and products
- Reference specific bank policies and current rates
- Be transparent about fees and performance expectations
- Maintain professional boundaries while being empathetic
- For sensitive financial matters, suggest in-person consultation when appropriate
- Store important details for future reference
"""
    
    def set_client_context(self, client_id: str) -> bool:
        """
        Set the client context for subsequent interactions.
        Loads all client information from database.
        
        Args:
            client_id: Client identifier
        
        Returns:
            Boolean indicating success
        """
        self.current_client_id = client_id
        client_profile = load_client_profile(client_id, self.db_connection)
        
        if client_profile.get("status") == "not_found":
            print(f"❌ Client {client_id} not found")
            return False
        
        print(f"✅ Client context loaded: {client_profile.get('name', 'Unknown')}")
        return True
    
    def start_conversation(self) -> str:
        """
        Start a new conversation thread for current client.
        
        Returns:
            Thread ID
        """
        if not self.current_client_id:
            raise ValueError("Client context not set. Call set_client_context() first")
        
        self.current_thread_id = create_client_thread(
            self.current_client_id, 
            self.db_connection
        )
        print(f"✅ Started new conversation thread: {self.current_thread_id}")
        return self.current_thread_id
    
    def build_context_for_response(self) -> str:
        """
        Build comprehensive context for LLM response.
        Includes: client profile, conversation history, bank knowledge.
        
        Returns:
            Context string for LLM prompt
        """
        # Get client profile and bank knowledge
        context = build_banking_context(self.current_client_id, self.db_connection)
        
        # Get recent conversation history
        history = retrieve_conversation_history(
            self.current_client_id,
            self.current_thread_id,
            self.db_connection,
            limit=20
        )
        
        context += "\n## RECENT CONVERSATION HISTORY\n"
        if history:
            for msg in history[-10:]:  # Last 10 messages
                role = msg['role'].upper()
                content = msg['content'][:500]  # Truncate long messages
                context += f"[{role}] {content}\n"
        else:
            context += "[Beginning of conversation]\n"
        
        return context
    
    def get_response(self, user_message: str) -> str:
        """
        Get AI response to user message with full context awareness.
        
        Args:
            user_message: User's input message
        
        Returns:
            Agent's response
        """
        if not self.current_client_id or not self.current_thread_id:
            raise ValueError("Client context and thread must be set")
        
        # Save user message to memory
        save_conversation_memory(
            self.current_client_id,
            self.current_thread_id,
            "user",
            user_message,
            self.db_connection
        )
        
        # Build rich context
        context = self.build_context_for_response()
        
        # Create prompt with context
        messages = [
            {
                "role": "system",
                "content": self.system_prompt + "\n\n" + context
            },
            {
                "role": "user",
                "content": user_message
            }
        ]
        
        # Get response from OpenAI
        try:
            response = self.openai_client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=0.7,
                max_tokens=1500
            )
            
            assistant_message = response.choices[0].message.content
            
            # Save assistant response to memory
            save_conversation_memory(
                self.current_client_id,
                self.current_thread_id,
                "assistant",
                assistant_message,
                self.db_connection
            )
            
            return assistant_message
            
        except Exception as e:
            logger.error(f"Error getting OpenAI response: {e}")
            return f"I apologize, but I encountered an error: {str(e)}"
    
    def get_conversation_summary(self, limit: int = 50) -> str:
        """
        Get summary of conversation history.
        
        Args:
            limit: Number of messages to include
        
        Returns:
            Formatted conversation summary
        """
        history = retrieve_conversation_history(
            self.current_client_id,
            self.current_thread_id,
            self.db_connection,
            limit=limit
        )
        
        if not history:
            return "No conversation history available"
        
        summary = f"Conversation Thread: {self.current_thread_id}\n"
        summary += f"Client: {self.current_client_id}\n"
        summary += f"Messages: {len(history)}\n\n"
        
        for msg in history:
            role = msg['role'].upper()
            timestamp = msg['timestamp']
            content = msg['content'][:200]  # Truncate
            summary += f"[{timestamp}] [{role}] {content}...\n"
        
        return summary

### Memory Stores and Managers

We configure seven memory types that our agent will use:

| Memory Type | Purpose |
|-------------|---------|
| **Conversational** | Chat history for context continuity |
| **Semantic (Knowledge Base)** | Documents and facts retrieved by similarity |
| **Workflow** | Past tool execution patterns |
| **Toolbox** | Available tools with semantic search |
| **Entity** | Extracted people, places, concepts |
| **Summary** | Compressed conversation summaries |
| **Tool Log** | Raw tool-call inputs, outputs, status, and errors for audit/JIT retrieval |

In [11]:
# Table names for each memory type
CONVERSATIONAL_TABLE = "CONVERSATIONAL_MEMORY"
KNOWLEDGE_BASE_TABLE = "SEMANTIC_MEMORY"
WORKFLOW_TABLE = "WORKFLOW_MEMORY"
TOOLBOX_TABLE = "TOOLBOX_MEMORY"
ENTITY_TABLE = "ENTITY_MEMORY"
SUMMARY_TABLE = "SUMMARY_MEMORY"
TOOL_LOG_TABLE = "TOOL_LOG_MEMORY"

### Clean Slate: Drop Existing Tables

<p style="background-color:#ff9a94; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note:</b> To ensure this lesson runs correctly regardless of whether previous lessons have been executed, we drop all memory tables before recreating them by running the cell below. This guarantees a clean starting state with consistent distance strategy and no stale data for the lesson.</p>

In [12]:
ALL_TABLES = [
    CONVERSATIONAL_TABLE,
    KNOWLEDGE_BASE_TABLE,
    WORKFLOW_TABLE,
    TOOLBOX_TABLE,
    ENTITY_TABLE,
    SUMMARY_TABLE,
    TOOL_LOG_TABLE]

# Drop existing tables to start fresh
for table in ALL_TABLES:
    try:
        with database_connection.cursor() as cur:
            cur.execute(f"DROP TABLE {table} PURGE")
            print(f"  - {table} (dropped)")
    except Exception as e:
        if "ORA-00942" in str(e):
            print(f"  - {table} (not exists)")
        else:
            print(f"  ✗ {table}: {e}")

database_connection.commit()

  - CONVERSATIONAL_MEMORY (dropped)
  - SEMANTIC_MEMORY (not exists)
  - WORKFLOW_MEMORY (not exists)
  - TOOLBOX_MEMORY (not exists)
  - ENTITY_MEMORY (not exists)
  - SUMMARY_MEMORY (not exists)
  - TOOL_LOG_MEMORY (dropped)


In [13]:

# Create or retrieve the conversational history table
from helper import create_conversational_history_table, create_tool_log_table
CONVERSATION_HISTORY_TABLE = create_conversational_history_table(database_connection, CONVERSATIONAL_TABLE)
TOOL_LOG_HISTORY_TABLE = create_tool_log_table(database_connection, TOOL_LOG_TABLE)

  ✅ Table CONVERSATIONAL_MEMORY created successfully with indexes
  ✅ Table TOOL_LOG_MEMORY created successfully with indexes


In [14]:
from langchain_oracledb.vectorstores import OracleVS
from langchain_community.vectorstores.utils import DistanceStrategy
from helper import StoreManager

# Create StoreManager instance
store_manager = StoreManager(
    client=database_connection,
    embedding_function=embedding_model,
    table_names={
        'knowledge_base': KNOWLEDGE_BASE_TABLE,
        'workflow': WORKFLOW_TABLE,
        'toolbox': TOOLBOX_TABLE,
        'entity': ENTITY_TABLE,
        'summary': SUMMARY_TABLE,
    },
    distance_strategy=DistanceStrategy.EUCLIDEAN_DISTANCE,
    conversational_table=CONVERSATION_HISTORY_TABLE,
    tool_log_table=TOOL_LOG_HISTORY_TABLE,
)

# Get all stores via the manager
conversation_table = store_manager.get_conversational_table()
knowledge_base_vs = store_manager.get_knowledge_base_store()
workflow_vs = store_manager.get_workflow_store()
toolbox_vs = store_manager.get_toolbox_store()
entity_vs = store_manager.get_entity_store()
summary_vs = store_manager.get_summary_store()
tool_log_table = store_manager.get_tool_log_table()

print("✅ All stores loaded via StoreManager")

✅ All stores loaded via StoreManager


In [15]:
from helper import MemoryManager, Toolbox, register_common_tools

# Initialize the MemoryManager instance
memory_manager = MemoryManager(
    conn=database_connection,
    conversation_table=conversation_table,
    knowledge_base_vs=knowledge_base_vs,
    workflow_vs=workflow_vs,
    toolbox_vs=toolbox_vs,
    entity_vs=entity_vs,
    summary_vs=summary_vs,
    tool_log_table=TOOL_LOG_HISTORY_TABLE
)

# Initialize Toolbox with embedding function
toolbox = Toolbox(memory_manager, model, embedding_model)

# Register common tools (arxiv search, paper fetch, etc.)
common_tools = register_common_tools(toolbox, memory_manager, KNOWLEDGE_BASE_TABLE)

print("✅ MemoryManager and Toolbox initialized")

✅ Registered 2 summary tools: ['expand_summary', 'summarize_and_store']
✅ Registered 5 common tools: ['arxiv_search_candidates', 'fetch_and_save_paper_to_kb_db', 'get_current_time', 'expand_summary', 'summarize_and_store']
✅ MemoryManager and Toolbox initialized


**Part 1 Takeaway:** We now have all memory infrastructure in place. The `MemoryManager` provides read/write access to all memory types, and the `Toolbox` has common tools registered for the agent to use.

---

## Part 2: Context Engineering Techniques

Context engineering is the practice of dynamically constructing the optimal input for an LLM based on the current query. Rather than passing everything to the model, we selectively retrieve and compress information to maximize relevance while staying within token limits.

### What This Section Covers

| Step | Function | Purpose |
|------|----------|---------|
| **1. Calculate Usage** | `calculate_context_usage()` | Monitor what % of the context window is used |
| **2. Summarize** | `summarise_context_window()` | Compress long content into summaries using LLM |
| **3. Offload** | `offload_to_summary()` | Auto-trigger summarization when usage exceeds threshold |
| **4. Just-in-Time Retrieval** | `expand_summary()` tool | Let agent expand summaries on demand |

**`Just-In-Time (JIT)`** retrieval is the process of fetching only the information needed at the exact moment the agent requires it, based on the current task, query, or reasoning step. Instead of loading pre-computed or pre-cached context upfront, the system dynamically retrieves the minimal, most relevant data on demand, ensuring efficiency and reducing context overload. In the context of agent memory JIT is a retrieval-control strategy where memory access is triggered by the agent’s current goal, query, or reasoning step. Rather than preloading large histories or the full knowledge base, the system dynamically filters, ranks, and injects only the information that materially influences the next token. This reduces context saturation, improves attention allocation, and increases reasoning fidelity.

In [16]:
# Import context window management functions from helper
# Summary tools are now loaded through register_common_tools in Part 1.
from helper import (
    calculate_context_usage,
    monitor_context_window,
    summarise_context_window,
    offload_to_summary,
    summarize_conversation,
)

print("✅ Context management functions loaded from helper.py")



✅ Context management functions loaded from helper.py


**Part 2 Takeaway:** Context engineering functions are now loaded from `helper.py`. These enable the agent to monitor context usage, summarize when needed, and expand summaries on demand.

---

## Part 3: The Memory-Aware Agent Loop

This is where everything comes together. The agent loop orchestrates memory operations at each step:

```
User Query
    ↓
1. BUILD CONTEXT — Read from all memory stores
    ↓
2. CHECK USAGE — Monitor token count, summarize if >80%
    ↓
3. SELECT TOOLS — Semantic search for relevant tools
    ↓
4. EXECUTE — LLM reasoning + tool calls
    ↓
5. PERSIST — Save conversation, workflow, entities
    ↓
Final Answer
```

### System Prompt and Tool Execution

In [17]:
import json as json_lib

AGENT_SYSTEM_PROMPT = """
# Role
You are a memory-aware agentic research assistant with access to tools.

# Context Window Structure (Partitioned Segments)
The user input is a partitioned context window. It contains a `# Question` section followed by memory segments.
Treat each segment as a distinct memory store with a specific purpose:
- `## Conversation Memory`
- `## Knowledge Base Memory`
- `## Workflow Memory`
- `## Entity Memory`
- `## Summary Memory`

# Memory Store Semantics
- Conversation Memory: Recent thread-level dialogue and instructions. Use it for continuity, user preferences, and unresolved requests.
- Knowledge Base Memory: Retrieved documents/passages. Use it to ground factual and technical claims.
- Workflow Memory: Prior execution patterns and step sequences. Use it to plan tool usage; adapt patterns, do not copy blindly.
- Entity Memory: Named people/orgs/systems and descriptors. Use it to disambiguate references and keep naming consistent.
- Summary Memory: Compressed older context represented by summary IDs. When thread-scoped summaries exist, prefer summaries for the active thread_id.

# Summary Expansion Policy
If critical detail is only present in Summary Memory or appears ambiguous, call `expand_summary(summary_id)` before relying on it.

# Operating Rules
1. Start with the provided memory segments before using tools.
2. If segments conflict, prioritize: current `# Question` > latest Conversation Memory > Knowledge Base evidence > older summaries/workflows.
3. Use only the tools provided in this turn and choose the minimum necessary tool calls.
4. If memory is insufficient, state what is missing and then use an appropriate tool.
5. For conversation compaction, use `summarize_and_store` with `thread_id` so source conversation units are marked as summarized.
"""


# def execute_tool(tool_name: str, tool_args: dict, current_thread_id: str | None = None) -> str:
#     """Execute a tool by looking it up in the toolbox."""

#     if tool_name not in toolbox._tools_by_name:
#         return f"Error: Tool '{tool_name}' not found"

#     args = dict(tool_args or {})

#     # Ensure conversation summarization marks source rows in the active thread.
#     if tool_name == "summarize_and_store" and "thread_id" not in args and current_thread_id is not None:
#         args["thread_id"] = str(current_thread_id)

#     return str(toolbox._tools_by_name[tool_name](**args) or "Done")

# # ==================== OPENAI CHAT FUNCTION ====================
# def call_openai_chat(messages: list, tools: list = None, model: str = "gpt-5-mini"):
#     """Call OpenAI Chat Completions API with tools."""
#     kwargs = {"model": model, "messages": messages}
#     if tools:
#         kwargs["tools"] = tools
#         kwargs["tool_choice"] = "auto"
#     return client.chat.completions.create(**kwargs)


def execute_tool(tool_name: str, tool_args: dict, current_thread_id: str | None = None) -> str:
    """Execute a tool by looking it up in the toolbox."""
    if tool_name not in toolbox._tools_by_name:
        return f"Error: Tool '{tool_name}' not found"

    args = dict(tool_args or {})

    # Ensure conversation summarization marks source rows in the active thread.
    if tool_name == "summarize_and_store" and "thread_id" not in args and current_thread_id is not None:
        args["thread_id"] = str(current_thread_id)

    # Execute and handle potential Gemini/Python object returns
    try:
        result = toolbox._tools_by_name[tool_name](**args)
        return str(result) if result is not None else "Done"
    except Exception as e:
        return f"Error executing tool '{tool_name}': {str(e)}"

# ==================== GEMINI CHAT FUNCTION ====================
def call_gemini_chat(prompt: str, history: list = None, tools: list = None):
    """
    Call Gemini GenerativeModel with tools and history.
    
    Args:
        prompt: The current user message/question.
        history: A list of previous turns in Gemini format: 
                 [{"role": "user", "parts": ["hi"]}, {"role": "model", "parts": ["hello"]}]
        tools: The actual function objects (not JSON schemas) or a Toolbox instance.
    """
    # 1. Initialize the chat session with history and system instruction
    # Note: In Gemini, system_instruction is passed to the GenerativeModel constructor
    # If not already there, we use a chat session.
    chat = llm_client.start_chat(history=history or [])

    # 2. Generate content
    # In Gemini, tools are usually bound to the model during initialization
    response = chat.send_message(prompt)

    return response

def handle_gemini_response(response, current_thread_id: str | None = None):
    """
    Helper to process Gemini's response, execute tools if requested, 
    and return the final text.
    """
    # Gemini handles multi-turn tool calling automatically if using start_chat,
    # but if you need to manualy execute:
    for part in response.candidates[0].content.parts:
        if part.function_call:
            fn_name = part.function_call.name
            fn_args = dict(part.function_call.args)
            
            # Execute tool
            observation = execute_tool(fn_name, fn_args, current_thread_id)
            
            # Note: To continue the loop in Gemini, you'd send the function_response 
            # back to the chat. Usually, Gemini's high-level SDK handles this 
            # if 'enable_automatic_function_calling=True' is set.
            return observation 

    return response.text




### Agent Loop: End-to-End Execution Flow

`call_agent()` is the orchestration layer that turns memory + tools into a reliable multi-step agent run.

1. **Build a partitioned context window**  
   The function pulls memory segments in order: Conversation, Knowledge Base, Workflow, Entity, and Summary.  
   This gives the model a structured, role-specific context instead of one mixed block.

2. **Protect context budget before reasoning**  
   Token usage is measured with `calculate_context_usage()`.  
   If usage is above 80%, `offload_to_summary()` compresses conversation-heavy content into Summary Memory and keeps summary references.

3. **Retrieve only relevant tools**  
   Toolbox memory is searched semantically (`read_toolbox(query, k=5)`), so the model sees a focused toolset for this query.

4. **Run iterative LLM-tool execution**  
   The model can issue tool calls, each tool runs, and the result is returned to the model through `role="tool"` messages.  
   Full raw tool outputs are persisted to `TOOL_LOG_MEMORY` for audit and later retrieval.

5. **Control tool-output bloat in prompt context**  
   The next LLM turn receives only the immediate tool result (truncated when very large), while the complete payload remains in the database.

6. **Persist learning artifacts**  
   At the end, the loop writes conversational turns, workflow steps, and extracted entities so future runs can reuse this execution history.



In [18]:
# ==================== MAIN AGENT LOOP ====================
def call_agent(query: str, thread_id: str = "1", max_iterations: int = 10) -> str:
    """Agent loop with context window monitoring and summarization."""
    thread_id = str(thread_id)
    steps = []
    summaries = []  # Track created summaries
    
    # 1. Build context from memory
    print("\n" + "="*50)
    print("🧠 BUILDING CONTEXT...")
    
    # Build memory context (excluding query for now)
    memory_context = ""
    memory_context += memory_manager.read_conversational_memory(thread_id) + "\n\n"
    memory_context += memory_manager.read_knowledge_base(query) + "\n\n"
    memory_context += memory_manager.read_workflow(query) + "\n\n"
    memory_context += memory_manager.read_entity(query) + "\n\n"
    memory_context += memory_manager.read_summary_context(query, thread_id=thread_id) + "\n\n"  # Shows IDs + descriptions (thread-scoped when available)
    
    # 2. Check context usage - summarize if >80%
    usage = calculate_context_usage(memory_context)
    print(f"📊 Context: {usage['percent']}% ({usage['tokens']}/{usage['max']} tokens)")
    
    if usage['percent'] > 80:
        print("⚠️ Context >80% - offloading conversation context to summary memory...")
        memory_context, summaries = offload_to_summary(
            memory_context,
            memory_manager,
            client,
            thread_id=thread_id,
        )
        if summaries:
            print(f"🧾 Created {len(summaries)} summary reference(s): {[s['id'] for s in summaries]}")
        usage = calculate_context_usage(memory_context)
        print(f"📊 After offload: {usage['percent']}% ({usage['tokens']}/{usage['max']} tokens)")
    
    # Now prepend the query (always preserved, never summarized)
    context = f"# Question\n{query}\n\n{memory_context}"

    print("====CONTEXT WINDOW=====\n")
    print(context)
    
    # 3. Get tools
    dynamic_tools = memory_manager.read_toolbox(query, k=5)
    print(f"🔧 Tools: {[t['function']['name'] for t in dynamic_tools]}")
    
    # 4. Store user message & extract entities
    memory_manager.write_conversational_memory(query, "user", thread_id)
    try:
        memory_manager.write_entity("", "", "", llm_client=client, text=query)
    except Exception:
        pass
    
    # 5. Agent loop
    messages = [{"role": "system", "content": AGENT_SYSTEM_PROMPT}, {"role": "user", "content": context}]
    final_answer = ""
    
    print("\n🤖 AGENT LOOP")
    for iteration in range(max_iterations):
        print(f"\n--- Iteration {iteration + 1} ---")
        
        response = call_openai_chat(messages, tools=dynamic_tools)
        msg = response.choices[0].message
        
        if msg.tool_calls:
            messages.append({"role": "assistant", "content": msg.content or "", "tool_calls": [
                {"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]})
            
            for tc in msg.tool_calls:
                tool_name = tc.function.name
                tool_args = json_lib.loads(tc.function.arguments)
                # Format args for display (truncate long values)
                args_display = {k: (v[:50] + '...' if isinstance(v, str) and len(v) > 50 else v) 
                               for k, v in tool_args.items()}
                print(f"🛠️ {tool_name}({args_display})")
                
                try:
                    result = execute_tool(tool_name, tool_args, current_thread_id=thread_id)
                    status = "success"
                    error_message = None
                    steps.append(f"{tool_name}({args_display}) → success")
                except Exception as e:
                    result = f"Error: {e}"
                    status = "failed"
                    error_message = str(e)
                    steps.append(f"{tool_name}({args_display}) → failed")

                # Persist full tool output to TOOL_LOG_MEMORY
                log_id = memory_manager.write_tool_log(
                    thread_id=thread_id,
                    tool_call_id=tc.id,
                    tool_name=tool_name,
                    tool_args=tool_args,
                    result=result,
                    status=status,
                    error_message=error_message,
                    metadata={"iteration": iteration + 1},
                )

                # Next call gets only the immediate tool result (bounded for context control)
                if len(result) > 3000:
                    result_for_llm = result[:3000] + f"\n\n[Truncated for context. Full output saved in TOOL_LOG_MEMORY as log_id: {log_id}]"
                else:
                    result_for_llm = result

                result_display = result_for_llm[:200] + "..." if len(result_for_llm) > 200 else result_for_llm
                print(f"   → {result_display}")
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result_for_llm})
        else:
            final_answer = msg.content or ""
            print(f"\n✅ DONE ({len(steps)} tool calls)")
            break
    else:
        # Max iterations reached without final answer
        print(f"\n⚠️ WARNING: Max iterations ({max_iterations}) reached without final answer")
        final_answer = "I was unable to complete the request within the allowed iterations."
    
    # 6. Save workflow & entities
    if steps:
        memory_manager.write_workflow(query, steps, final_answer)
    try:
        memory_manager.write_entity("", "", "", llm_client=client, text=final_answer)
    except Exception:
        pass
    memory_manager.write_conversational_memory(final_answer, "assistant", thread_id)
    
    print("\n" + "="*50 + f"\n💬 ANSWER:\n{final_answer}\n" + "="*50)
    return final_answer



In [19]:
# ==================== MAIN AGENT LOOP (GEMINI VERSION) ====================
def call_agent(query: str, thread_id: str = "1", max_iterations: int = 10) -> str:
    """Agent loop using Gemini Chat Sessions with context monitoring."""
    thread_id = str(thread_id)
    steps = []
    
    # 1. Build context from memory
    print("\n" + "="*50)
    print("🧠 BUILDING CONTEXT...")
    
    memory_context = ""
    memory_context += memory_manager.read_conversational_memory(thread_id) + "\n\n"
    memory_context += memory_manager.read_knowledge_base(query) + "\n\n"
    memory_context += memory_manager.read_workflow(query) + "\n\n"
    memory_context += memory_manager.read_entity(query) + "\n\n"
    memory_context += memory_manager.read_summary_context(query, thread_id=thread_id) + "\n\n"
    
    # 2. Check context usage
    usage = calculate_context_usage(memory_context)
    print(f"📊 Context: {usage['percent']}% ({usage['tokens']}/{usage['max']} tokens)")
    
    if usage['percent'] > 80:
        print("⚠️ Context >80% - offloading to summary memory...")
        # Note: Ensure offload_to_summary is also updated to use llm_client.generate_content()
        memory_context, summaries = offload_to_summary(
            memory_context,
            memory_manager,
            llm_client, # Changed from client
            thread_id=thread_id,
        )
        usage = calculate_context_usage(memory_context)
    
    # Prepend the query
    context = f"# Question\n{query}\n\n{memory_context}"
    print("====CONTEXT WINDOW=====\n")
    print(context)
    
    # 3. Handle Tools
    # Note: For Gemini, it's best to initialize the model with ALL tools once.
    # If you must use dynamic tools, you'd re-initialize the model object here.
    print(f"🔧 Tools active in toolbox: {list(toolbox._tools_by_name.keys())}")
    
    # 4. Store user message & extract entities
    memory_manager.write_conversational_memory(query, "user", thread_id)
    try:
        memory_manager.write_entity("", "", "", llm_client=llm_client, text=query)
    except Exception: pass
    
    # 5. Gemini Agent Loop
    # We use start_chat to manage the conversation history automatically
    chat = llm_client.start_chat(history=[])
    final_answer = ""
    
    print("\n🤖 AGENT LOOP (Gemini)")
    # Send the combined system prompt + context as the first message
    current_prompt = f"{AGENT_SYSTEM_PROMPT}\n\n{context}"
    
    for iteration in range(max_iterations):
        print(f"\n--- Iteration {iteration + 1} ---")
        
        # Call Gemini
        response = chat.send_message(current_prompt)
        
        # Check for Function Calls in the parts
        found_tool_call = False
        for part in response.candidates[0].content.parts:
            if part.function_call:
                found_tool_call = True
                tool_name = part.function_call.name
                tool_args = dict(part.function_call.args)
                
                # Truncate args for display
                args_display = {k: (v[:50] + '...' if isinstance(v, str) and len(v) > 50 else v) 
                               for k, v in tool_args.items()}
                print(f"🛠️ {tool_name}({args_display})")
                
                try:
                    result = execute_tool(tool_name, tool_args, current_thread_id=thread_id)
                    status = "success"
                    steps.append(f"{tool_name} → success")
                except Exception as e:
                    result = f"Error: {e}"
                    status = "failed"
                    steps.append(f"{tool_name} → failed")

                # Log tool execution
                log_id = memory_manager.write_tool_log(
                    thread_id=thread_id,
                    tool_call_id="gemini_call", # Gemini doesn't always provide a call_id like OpenAI
                    tool_name=tool_name,
                    tool_args=tool_args,
                    result=result,
                    status=status
                )
                
                # Truncate result for next turn
                result_for_llm = result[:3000] if len(result) > 3000 else result
                print(f"    → {result_for_llm[:200]}...")
                
                # In Gemini's chat session, the next "send_message" should contain 
                # the function response. 
                current_prompt = [
                    genai.protos.Content(
                        role="user",
                        parts=[genai.protos.Part(
                            function_response=genai.protos.FunctionResponse(
                                name=tool_name,
                                response={'result': result_for_llm}
                            )
                        )]
                    )
                ]
                break # Gemini usually handles one call or a bundle; break to send response

        if not found_tool_call:
            final_answer = response.text
            print(f"\n✅ DONE ({len(steps)} tool calls)")
            break
    else:
        print(f"\n⚠️ WARNING: Max iterations reached")
        final_answer = "I couldn't finish in time."

    # 6. Finalize Memory
    if steps:
        memory_manager.write_workflow(query, steps, final_answer)
    try:
        memory_manager.write_entity("", "", "", llm_client=llm_client, text=final_answer)
    except Exception: pass
    memory_manager.write_conversational_memory(final_answer, "assistant", thread_id)
    
    print("\n" + "="*50 + f"\n💬 ANSWER:\n{final_answer}\n" + "="*50)
    return final_answer

### Testing the Agent

Now let's test our memory-aware agent. Notice how it:
- Builds context from memory before responding
- Uses tools to fetch and store information
- Persists the interaction for future context

In [20]:
# 2. ADD these lines:
import google.generativeai as genai
import os

# 3. Configure and Initialize
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

# Now this will work:
llm_client = genai.GenerativeModel(
    model_name="gemini-2.5-flash-lite", # Use a stable name like 1.5-flash
    tools=list(toolbox._tools_by_name.values()),
    system_instruction=AGENT_SYSTEM_PROMPT
)

In [21]:
call_agent("i am shreyash, and i work in the field of ai", thread_id="50000")


🧠 BUILDING CONTEXT...
📊 Context: 0.2% (623/256000 tokens)
====CONTEXT WINDOW=====

# Question
i am shreyash, and i work in the field of ai

## Conversation Memory
### What this memory is
Chronological, unsummarized messages from the current thread. This memory captures user intent, constraints, and commitments made in recent turns.
### How you should leverage it
- Preserve continuity with prior decisions, terminology, and user preferences.
- Resolve references like "that", "previous step", or "the paper above" using earlier turns.
- If older context conflicts with newer user instructions, prioritize the latest user direction.
### Retrieved messages

(No unsummarized messages found for this thread.)

## Knowledge Base Memory
### What this memory is
Retrieved background documents and previously ingested reference material relevant to the current query.
### How you should leverage it
- Ground responses in these passages when making factual or technical claims.
- Prefer concrete details f

"Hi Shreyash! It's nice to meet you. How can I help you with your AI work today?"

In [20]:
call_agent("hi, i am rahul, i love cricket", thread_id="50000")


🧠 BUILDING CONTEXT...
📊 Context: 0.3% (705/256000 tokens)
====CONTEXT WINDOW=====

# Question
hi, i am rahul, i love cricket

## Conversation Memory
### What this memory is
Chronological, unsummarized messages from the current thread. This memory captures user intent, constraints, and commitments made in recent turns.
### How you should leverage it
- Preserve continuity with prior decisions, terminology, and user preferences.
- Resolve references like "that", "previous step", or "the paper above" using earlier turns.
- If older context conflicts with newer user instructions, prioritize the latest user direction.
### Retrieved messages

[15:24:39] [user] hi
[15:24:45] [assistant] Hello! How can I assist you today? I am a research assistant with access to tools for searching arXiv, fetching and processing technical papers, and managing project information. 

Whether you need help finding specific literature, summarizing complex topics, or organizing your research workflow, I'm here to h

"Nice to meet you, Rahul! It's great to know you're a cricket fan.\n\nAs a research assistant, I'm here to help you dive into any topic you're curious about. If you ever want to explore the more technical side of your interest—such as sports analytics, the physics of ball swing, or even the impact of technology on the game—I can find and process research papers for you.\n\nHow can I assist you with your research or projects today?"

In [ ]:
call_agent("What is my name?", thread_id="50000")

In [ ]:
call_agent("Summarize the converstation so far using your tool", thread_id="50000")

In [ ]:
call_agent("What was my first question?", thread_id="50000")

**Part 3 Takeaway:** The agent loop demonstrates how memory operations integrate into the execution flow—reading context before reasoning, managing token limits dynamically, and persisting results for future use.

---

## Lesson Summary

In this lesson, you built a complete **Memory Aware Agent** that:

| Capability | Implementation |
|------------|----------------|
| **Reads Memory** | Retrieves from 7 memory types before each response (tool logs remain JIT by default) |
| **Manages Context** | Monitors tokens, summarizes when >80% capacity |
| **Uses Tools** | Semantic search selects relevant tools per query |
| **Persists Learning** | Saves conversations, workflows, entities, and raw tool logs |
| **Expands On-Demand** | JIT retrieval via `expand_summary()` tool |

**Key Insight:** A memory-aware agent doesn't just respond to queries—it *learns* from each interaction. Information discovered, decisions made, and patterns executed are all persisted, making the agent more capable over time.